# 기프트코 FAQ RAG v1 — 임베딩 기반 검색

TF-IDF(키워드 매칭) → **임베딩 의미 검색**으로 업그레이드하는 노트북입니다.

## 실행 흐름
1. FAQ 데이터 로드 (보완 v1, 38개 항목)
2. TF-IDF 한계 확인 (기존 방식의 실패 케이스)
3. 임베딩 모델로 FAQ 인코딩
4. 의미 기반 검색 함수 구현
5. TF-IDF vs 임베딩 비교 실험
6. LLM으로 최종 답변 생성 (Claude API)

## 0. 패키지 설치

In [ ]:
# 처음 실행 시에만 필요합니다
# !pip install -q sentence-transformers anthropic scikit-learn pandas openpyxl numpy

## 1. 라이브러리 불러오기 & FAQ 데이터 로드

In [ ]:
import os
import pickle
import requests
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sk_cosine
from sentence_transformers import SentenceTransformer

FAQ_PATH = Path('./data/기프트코_FAQ_보완_v1.xlsx')
CACHE_DIR = Path('./cache')  # 임베딩 캐시 — 기법(모델) 단위로 저장, 버전 폴더에 종속되지 않음
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# 원본 로드 & 답변 없는 항목 제외
raw = pd.read_excel(FAQ_PATH, dtype=str).fillna('')
rag_df = raw[raw['답변'].str.strip() != ''].copy().reset_index(drop=True)
rag_df['문서ID'] = [f'FAQ-{i+1:04d}' for i in range(len(rag_df))]

print(f'전체: {len(raw)}개  /  RAG 사용: {len(rag_df)}개 (답변 없는 {len(raw)-len(rag_df)}개 제외)')
rag_df[['문서ID', '구분', '질문']].head(10)

## 2. TF-IDF의 한계 먼저 확인

이전 v0 노트북의 TF-IDF 검색에서 실패했던 케이스를 재현합니다.

> 질문: **"시안 수정은 가능한가요?"**  
> 기대 결과: 시안 관련 FAQ  
> TF-IDF 실제 결과: 전혀 다른 FAQ가 1위

In [ ]:
# TF-IDF 인덱스 (비교용으로 새로 만들기)
def make_doc(row):
    return f"구분: {row['구분']}\n질문: {row['질문']}\n답변: {row['답변']}"

rag_df['문서'] = rag_df.apply(make_doc, axis=1)

tfidf_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), sublinear_tf=True)
tfidf_mat = tfidf_vec.fit_transform(rag_df['문서'].tolist())

def tfidf_search(query, top_k=3):
    q_vec = tfidf_vec.transform([query])
    scores = sk_cosine(q_vec, tfidf_mat).flatten()
    idx = scores.argsort()[::-1][:top_k]
    res = rag_df.iloc[idx].copy()
    res.insert(0, 'score', scores[idx])
    return res[['score', '문서ID', '구분', '질문']].reset_index(drop=True)

# 실패 케이스 확인
FAIL_QUERY = '시안 수정은 가능한가요?'
print(f'질문: {FAIL_QUERY}')
print('─' * 60)
print(tfidf_search(FAIL_QUERY).to_string())

In [ ]:
# 추가 실패 케이스 확인
for q in ['제작 취소할 수 있나요?', '배송 얼마나 걸리나요?', '파일 어떻게 보내나요?']:
    print(f'\n질문: {q}')
    top1 = tfidf_search(q, top_k=1)
    print(f'  TF-IDF 1위: [{top1.iloc[0]["구분"]}] {top1.iloc[0]["질문"]}  (score={top1.iloc[0]["score"]:.3f})')

## 3. 임베딩 모델 로드

**`jhgan/ko-sroberta-multitask`**  
- 한국어 특화 Sentence-BERT 모델
- 768차원 벡터, 약 480MB
- 의미가 비슷한 문장 → 벡터 거리가 가까움

> 이미 `BAAI/bge-m3`가 설치된 경우 `MODEL_NAME`을 변경해도 됩니다.

In [ ]:
MODEL_NAME = 'jhgan/ko-sroberta-multitask'
# MODEL_NAME = 'BAAI/bge-m3'  # 더 강력, 이미 다운로드된 경우 사용

print(f'모델 로딩: {MODEL_NAME}')
model = SentenceTransformer(MODEL_NAME)
print(f'임베딩 차원: {model.get_embedding_dimension()}')

## 4. FAQ 임베딩 생성

각 FAQ 항목의 **질문 텍스트**를 벡터로 변환합니다.

> 왜 질문만? 유저 입력과 FAQ 질문이 형태가 비슷하기 때문.  
> 질문+답변 전체를 인코딩하면 답변이 길수록 노이즈가 생깁니다.

In [ ]:
EMBED_PATH = CACHE_DIR / 'faq_embeddings_jhgan.npy'

if EMBED_PATH.exists():
    print('저장된 임베딩 로드...')
    faq_embeddings = np.load(EMBED_PATH)
else:
    print('임베딩 생성 중...')
    questions = rag_df['질문'].tolist()
    faq_embeddings = model.encode(questions, normalize_embeddings=True, show_progress_bar=True)
    np.save(EMBED_PATH, faq_embeddings)
    print(f'저장 완료: {EMBED_PATH}')

print(f'임베딩 shape: {faq_embeddings.shape}  (FAQ 수, 벡터 차원)')

In [ ]:
# 임베딩 직관적으로 이해하기 — 비슷한 질문은 코사인 유사도가 높다
q1 = '시안 수정은 몇 번 가능한가요?'
q2 = '시안을 몇 번이나 바꿀 수 있나요?'
q3 = '배송비는 얼마예요?'

vecs = model.encode([q1, q2, q3], normalize_embeddings=True)
sim12 = float(np.dot(vecs[0], vecs[1]))
sim13 = float(np.dot(vecs[0], vecs[2]))

print(f'유사한 질문 (q1 vs q2): {sim12:.4f}')
print(f'다른 주제  (q1 vs q3): {sim13:.4f}')
print('→ 의미가 비슷할수록 코사인 유사도가 높습니다.')

## 5. 임베딩 검색 함수 구현

검색 원리:
```
사용자 질문 → 임베딩 → 벡터
FAQ 벡터들과 코사인 유사도 계산
유사도 높은 순으로 top-k 반환
```

In [ ]:
def embed_search(query: str, top_k: int = 3, min_score: float = 0.0):
    """임베딩 코사인 유사도 기반 FAQ 검색"""
    q_vec = model.encode([query], normalize_embeddings=True)  # shape: (1, dim)
    scores = np.dot(faq_embeddings, q_vec.T).flatten()        # (n_faq,)
    top_idx = scores.argsort()[::-1][:top_k]
    result = rag_df.iloc[top_idx].copy()
    result.insert(0, 'score', scores[top_idx])
    result = result[result['score'] >= min_score]
    return result[['score', '문서ID', '구분', '질문', '답변']].reset_index(drop=True)

def print_embed_result(query, top_k=3):
    res = embed_search(query, top_k)
    print(f'질문: {query}')
    print('=' * 70)
    for _, row in res.iterrows():
        print(f"  score={row['score']:.4f}  [{row['구분']}] {row['질문']}")
    print()
    return res

# 기본 동작 확인
print_embed_result('시안 수정은 가능한가요?')

## 6. TF-IDF vs 임베딩 — 나란히 비교

**핵심 관찰 포인트**
- TF-IDF: 질문에 쓰인 단어가 문서에 없으면 실패
- 임베딩: 표현이 달라도 의미가 같으면 잘 찾음

In [ ]:
def compare(query, top_k=1):
    tf_res  = tfidf_search(query, top_k)
    emb_res = embed_search(query, top_k)

    print(f'질문: {query}')
    print(f'  TF-IDF  1위: [{tf_res.iloc[0]["구분"]}] {tf_res.iloc[0]["질문"]}')
    print(f'  임베딩  1위: [{emb_res.iloc[0]["구분"]}] {emb_res.iloc[0]["질문"]}')
    print()

test_queries = [
    '시안 수정은 가능한가요?',
    '제작 취소할 수 있나요?',
    '파일 어떻게 보내나요?',
    '배송 얼마나 걸리나요?',
    '영수증 다시 받을 수 있나요?',
    '단체 주문 어떻게 해요?',
    '처음 주문인데 순서가 어떻게 되나요?',
]

for q in test_queries:
    compare(q)

### 6-2. Ollama 모델별 임베딩 검색 비교

Ollama에 설치된 각 모델로 FAQ를 임베딩했을 때 검색 결과가 어떻게 달라지는지 비교합니다.

| 모델 유형 | 특징 |
|-----------|------|
| `bge-m3` | 임베딩 전용 모델 — 의미 검색에 최적화 |
| `qwen2.5`, `gemma3` 등 | 생성형 LLM — 임베딩 추출은 가능하나 전용 모델보다 품질이 낮을 수 있음 |

> **참고**: LLM 모델은 임베딩 추출 시 전체 파라미터를 로드하므로 모델 크기에 따라 수 분이 걸릴 수 있습니다.

In [ ]:
import time

# 비교할 모델 목록 — 시간이 오래 걸리면 일부만 남기고 주석 처리
COMPARE_MODELS = [
    'bge-m3:latest',
    'qwen2.5:3b',
    'qwen2.5:7b',
    'gemma3:4b',
    'gemma2:9b',
    'gemma3:12b',
]

# 비교에 사용할 테스트 질문
COMPARE_QUERIES = [
    '시안 수정은 가능한가요?',
    '제작 취소할 수 있나요?',
    '파일 어떻게 보내나요?',
    '배송 얼마나 걸리나요?',
    '영수증 다시 받을 수 있나요?',
    '단체 주문 어떻게 해요?',
]

def ollama_embed_batch(texts: list[str], model: str, base_url: str = 'http://localhost:11434') -> np.ndarray:
    """Ollama /api/embed 로 여러 텍스트를 한 번에 임베딩"""
    resp = requests.post(
        f'{base_url}/api/embed',
        json={'model': model, 'input': texts},
        timeout=600,
    )
    resp.raise_for_status()
    return np.array(resp.json()['embeddings'], dtype=np.float32)

def cosine_norm(vecs: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vecs, axis=1, keepdims=True)
    return vecs / np.where(norms == 0, 1.0, norms)

def search_with_model(model: str, query: str, questions: list[str], faq_vecs_normed: np.ndarray, top_k: int = 1):
    q_vec = cosine_norm(ollama_embed_batch([query], model))  # (1, dim)
    scores = np.dot(faq_vecs_normed, q_vec.T).flatten()
    idx = scores.argsort()[::-1][:top_k]
    return [(scores[i], rag_df.iloc[i]['구분'], rag_df.iloc[i]['질문']) for i in idx]

# ── 모델별 FAQ 임베딩 생성 ─────────────────────────────────
questions = rag_df['질문'].tolist()
model_faq_vecs = {}  # model -> normalized embeddings

for m in COMPARE_MODELS:
    t0 = time.time()
    print(f'  임베딩 생성: {m} ... ', end='', flush=True)
    try:
        vecs = cosine_norm(ollama_embed_batch(questions, m))
        model_faq_vecs[m] = vecs
        print(f'완료 ({time.time()-t0:.1f}s, dim={vecs.shape[1]})')
    except Exception as e:
        print(f'실패 — {e}')
        model_faq_vecs[m] = None

print('\n완료된 모델:', [m for m, v in model_faq_vecs.items() if v is not None])

In [ ]:
# ── 질문별 검색 결과 비교표 ────────────────────────────────
rows = []
for q in COMPARE_QUERIES:
    row = {'질문': q}
    # TF-IDF 기준선
    tf_top = tfidf_search(q, top_k=1).iloc[0]
    row['TF-IDF'] = f"[{tf_top['구분']}]\n{tf_top['질문']}"
    # Ollama 각 모델
    for m in COMPARE_MODELS:
        if model_faq_vecs[m] is None:
            row[m] = '실패'
            continue
        q_vec = cosine_norm(ollama_embed_batch([q], m))
        scores = np.dot(model_faq_vecs[m], q_vec.T).flatten()
        top_idx = scores.argmax()
        hit = rag_df.iloc[top_idx]
        row[m] = f"[{hit['구분']}]\n{hit['질문']}\n(score={scores[top_idx]:.3f})"
    rows.append(row)

result_df = pd.DataFrame(rows).set_index('질문')

# 출력
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 50)

print('=== 모델별 FAQ 검색 결과 비교 (각 셀: [카테고리] 검색된 FAQ 질문) ===\n')
for q in COMPARE_QUERIES:
    print(f'▶ 입력 질문: {q}')
    print(f'  TF-IDF  : {rows[COMPARE_QUERIES.index(q)]["TF-IDF"].replace(chr(10), " ")}')
    for m in COMPARE_MODELS:
        if model_faq_vecs.get(m) is not None:
            val = rows[COMPARE_QUERIES.index(q)].get(m, '실패').replace(chr(10), ' ')
            short_m = m.replace(':latest', '')
            print(f'  {short_m:<18}: {val}')
    print()

### 6-3. 한국어 Sentence-Transformer 모델 비교

Ollama 모델들과는 별개로, **sentence-transformers 한국어 모델** 2개를 비교합니다.

| 모델 | 베이스 | 학습 데이터 |
|------|--------|------------|
| `jhgan/ko-sroberta-multitask` | RoBERTa | KorSTS + KorNLI |
| `snunlp/KR-SBERT-V40K-klueNLI-augSTS` | BERT | KLUE-NLI + augSTS |

두 모델 모두 768차원이며 `sentence-transformers`로 바로 사용 가능합니다.

In [ ]:
from sentence_transformers import SentenceTransformer

# 비교할 sentence-transformer 모델 목록
ST_MODELS = {
    'jhgan (현재)':  'jhgan/ko-sroberta-multitask',
    'KR-SBERT':      'snunlp/KR-SBERT-V40K-klueNLI-augSTS',
}

# 모델별 캐시 파일 — 기법(모델) 기준으로 고정된 이름 사용 (버전에 종속되지 않음)
ST_CACHE_FILES = {
    'jhgan (현재)':  CACHE_DIR / 'faq_embeddings_jhgan.npy',
    'KR-SBERT':      CACHE_DIR / 'faq_embeddings_krsbert.npy',
}

questions = rag_df['질문'].tolist()
st_faq_vecs   = {}   # model_label -> normalized FAQ embeddings
st_model_objs = {}   # model_label -> SentenceTransformer instance

for label, model_id in ST_MODELS.items():
    embed_cache = ST_CACHE_FILES[label]
    print(f'로딩: {label}  ({model_id})')
    m = SentenceTransformer(model_id)
    st_model_objs[label] = m

    if embed_cache.exists():
        vecs = np.load(embed_cache)
        print(f'  캐시 로드 완료  dim={vecs.shape[1]}')
    else:
        vecs = m.encode(questions, normalize_embeddings=True, show_progress_bar=True)
        np.save(embed_cache, vecs)
        print(f'  임베딩 생성 & 저장  dim={vecs.shape[1]}')

    st_faq_vecs[label] = vecs

In [ ]:
# ── 질문별 검색 결과 비교 ─────────────────────────────────
print('=== 한국어 Sentence-Transformer 모델 비교 ===\n')
print(f'{"질문":<22}  {"TF-IDF":<30}  ' + '  '.join(f'{lb:<30}' for lb in ST_MODELS))
print('─' * (22 + 2 + 32 * (1 + len(ST_MODELS))))

for q in COMPARE_QUERIES:
    # TF-IDF
    tf_top  = tfidf_search(q, top_k=1).iloc[0]
    tf_str  = f"[{tf_top['구분']}] {tf_top['질문']}"[:29]

    # 각 sentence-transformer 모델
    st_strs = []
    for label, faq_vecs in st_faq_vecs.items():
        m_obj  = st_model_objs[label]
        q_vec  = m_obj.encode([q], normalize_embeddings=True)
        scores = np.dot(faq_vecs, q_vec.T).flatten()
        top_i  = scores.argmax()
        hit    = rag_df.iloc[top_i]
        st_strs.append(f"[{hit['구분']}] {hit['질문']} ({scores[top_i]:.3f})"[:29])

    print(f'{q:<22}  {tf_str:<30}  ' + '  '.join(f'{s:<30}' for s in st_strs))

print()
print('※ TF-IDF는 단어 일치, 임베딩 모델은 의미 유사도 기반 검색입니다.')
print('  두 임베딩 모델의 결과가 다른 항목에서 어떤 모델이 더 적절한지 살펴보세요.')

## 7. LLM 답변 생성 (Ollama / Claude API / OpenRouter)

검색된 FAQ를 근거로 자연어 답변을 생성합니다.

```
사용자 질문
   → 임베딩 검색으로 관련 FAQ top-k 선택
   → FAQ 내용을 컨텍스트로 넣어 프롬프트 구성
   → LLM이 고객 응대 말투로 답변 생성
```

**우선순위**: Ollama(로컬) → Anthropic API → OpenRouter  
Ollama가 실행 중이면 API 키 없이 바로 사용 가능합니다.

**Ollama 준비:**
```bash
# 모델 설치 예시 (하나만 선택)
ollama pull qwen2.5          # 한국어 품질 좋음 (추천)
ollama pull llama3.2
ollama pull gemma3
```

In [ ]:
# ── Ollama 설정 (로컬, API 키 불필요) ──────────────────────
OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_MODEL    = 'qwen2.5:3b'   # ollama list 로 설치된 모델 확인 후 변경

# ── 클라우드 API (없으면 빈 문자열 유지) ──────────────────
ANTHROPIC_API_KEY  = os.getenv('ANTHROPIC_API_KEY', '')
OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY', '')

# Ollama 실행 여부 확인
def check_ollama() -> bool:
    try:
        r = requests.get(f'{OLLAMA_BASE_URL}/api/tags', timeout=3)
        models = [m['name'].split(':')[0] for m in r.json().get('models', [])]
        print(f'Ollama 실행 중 ✓  설치된 모델: {models}')
        return True
    except Exception:
        print('Ollama 미실행 (ollama serve 로 시작하거나 앱 실행)')
        return False

import requests
OLLAMA_AVAILABLE = check_ollama()
print()
print('사용할 LLM:',
      f'Ollama ({OLLAMA_MODEL})' if OLLAMA_AVAILABLE else
      'Anthropic API'            if ANTHROPIC_API_KEY  else
      'OpenRouter'               if OPENROUTER_API_KEY else
      '없음 (프롬프트만 출력)')

In [ ]:
def build_prompt(query: str, retrieved_df: pd.DataFrame) -> str:
    blocks = []
    for i, row in retrieved_df.iterrows():
        blocks.append(
            f"[근거 {i+1}] 구분: {row['구분']} / FAQ 질문: {row['질문']}\nFAQ 답변: {row['답변']}"
        )
    context = '\n\n'.join(blocks)

    return f"""당신은 기프트코 고객지원 FAQ 챗봇입니다.
아래 FAQ 근거만 사용해서 답변하세요.
FAQ에 없는 내용은 단정하지 말고 '고객센터로 문의해 주세요'라고 안내하세요.

[고객 질문]
{query}

[검색된 FAQ 근거]
{context}

[답변 작성 기준]
- 결론을 먼저 말하세요.
- FAQ 근거 내용만 사용하세요.
- 마지막에 참고 문서ID를 표시하세요.""".strip()

In [ ]:
def generate_answer(query: str, top_k: int = 3) -> dict:
    retrieved = embed_search(query, top_k=top_k)
    prompt = build_prompt(query, retrieved)
    system = '당신은 제공된 근거만 사용해서 답변하는 한국어 고객지원 챗봇입니다.'

    # ── 1순위: Ollama (로컬, API 키 불필요) ──────────────────
    if OLLAMA_AVAILABLE:
        resp = requests.post(
            f'{OLLAMA_BASE_URL}/api/chat',
            json={
                'model': OLLAMA_MODEL,
                'messages': [
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': prompt},
                ],
                'options': {
                    'temperature': 0.2
                },
                'stream': False,
            },
            timeout=120,
        )

        resp.raise_for_status()
        answer = resp.json()['message']['content']

    # ── 2순위: Anthropic Claude API ──────────────────────────
    elif ANTHROPIC_API_KEY:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        msg = client.messages.create(
            model='claude-haiku-4-5-20251001',
            max_tokens=1024,
            system=system,
            messages=[{'role': 'user', 'content': prompt}],
        )
        answer = msg.content[0].text

    # ── 3순위: OpenRouter ────────────────────────────────────
    elif OPENROUTER_API_KEY:
        resp = requests.post(
            'https://openrouter.ai/api/v1/chat/completions',
            headers={'Authorization': f'Bearer {OPENROUTER_API_KEY}'},
            json={
                'model': 'openai/gpt-4o-mini',
                'messages': [
                    {'role': 'system', 'content': system},
                    {'role': 'user', 'content': prompt},
                ],
                'temperature': 0.2,
            },
            timeout=60,
        )

        resp.raise_for_status()
        answer = resp.json()['choices'][0]['message']['content']

    # ── LLM 없음: 프롬프트만 출력 ───────────────────────────
    else:
        print('LLM이 설정되지 않았습니다. 아래는 LLM에 전달될 프롬프트입니다.')
        print('=' * 70)
        print(prompt)
        return {'answer': None, 'retrieved': retrieved, 'prompt': prompt}

    return {'answer': answer, 'retrieved': retrieved, 'prompt': prompt}

In [ ]:
# RAG 답변 테스트
test_q = '시안 수정은 몇 번이나 가능한가요?'
result = generate_answer(test_q, top_k=3)

if result['answer']:
    print(f'질문: {test_q}')
    print('─' * 70)
    print(result['answer'])
    print()
    print('참고된 FAQ:')
    print(result['retrieved'][['문서ID', '구분', '질문', 'score']].to_string())

In [ ]:
# 여러 질문 연속 테스트
questions = [
    '비회원도 주문할 수 있나요?',
    '제작 취소하려면 어떻게 해야 하나요?',
    '배송 현황은 어디서 확인하나요?',
]

for q in questions:
    r = generate_answer(q, top_k=3)
    print(f'Q: {q}')
    if r['answer']:
        print(f'A: {r["answer"][:200]}...')
    print()

## 다음 단계

이 노트북에서 배운 것:
- 임베딩이 TF-IDF보다 의미 검색에 왜 강한지
- `SentenceTransformer`로 문서를 벡터로 만드는 법
- 코사인 유사도로 가장 관련 있는 문서를 찾는 법
- 검색 결과를 LLM 프롬프트에 넣어 답변 생성하는 법

**RAG v2에서 더 발전시킬 것들:**

| 개선 포인트 | 방법 |
|-------------|------|
| 검색 정확도 높이기 | BM25(키워드) + 임베딩(의미) 하이브리드 |
| 속도 최적화 | FAISS 인덱스 사용 |
| 답변 품질 | 리랭킹(Cross-encoder) 추가 |
| 지식 확장 | 상품 데이터 + FAQ 합쳐서 검색 |

다음은 **하이브리드 검색(BM25 + 임베딩)** 을 실습해보면 좋습니다.